# 12. 질문 목적에 따른 검색과 연결된 근거 읽기

## 이번 실험
질문이 안내도 위치, 표의 수치·규격, 사용 설명 중 무엇을 찾는지 판단합니다.
검색어 일치 점수는 긴 부모 전체 대신 작은 청크에서 계산합니다.
안내도는 번호 항목의 부품 이름도 별도 색인으로 비교합니다.

**순위를 정하는 규칙에는 평가 질문의 정답 ID를 사용하지 않습니다.**
기존 30개 질문과 기대 출처는 그대로 두고 아래 세 방식을 비교합니다.

1. 기존 결합 검색: 부모 전체 글자 색인 + 의미 검색
2. 청크 결합 검색: 작은 조각 글자 색인 + 의미 검색
3. 유형 선택 검색: 필요한 자료 유형 안에서 청크 결합 검색

첫 결과와 상위 3개를 각각 비교합니다. 후보 수가 줄면 상위3에 포함되기 쉬워지므로
유형별 후보 수와 첫 결과도 함께 봅니다. 실제 생성 답변의 정확도를 평가하는 실험은 아닙니다.

전체 벡터 파일을 저장하지 않았으므로 첫 셀은 11번 준비·임베딩을 다시 실행합니다.
같은 커널에서 11번을 이미 끝냈다면 기존 메모리 자료를 재사용합니다.


## 수업 기반과 프로젝트 추가 기록 (2026-10-01)

코드의 추가 설명은 주석으로 남겼습니다. 이 표시는 프로젝트 적용 기록이며, 수업에서 개념 자체를 다루지 않았다는 뜻은 아닙니다.

- 작은 청크와 안내도 부품 항목의 글자 색인, 질문 목적 규칙으로 검색 후보를 선택합니다.
- 추가: 안내도·표·설명문 규칙, 의미/글자 점수 50% 결합, 후보 부족 시 전체 후보 보충, 연결 근거·검토 상태 반환.
- 별도 분류 모델이나 재순위 모델은 쓰지 않습니다. 규칙이 자료 밖 질문의 답변 가능 여부를 보장하지는 않습니다.

전체 목록: [수업 기반·추가 적용 기록](../src/zzong_santafe_lag/learning_additions.md)


In [1]:
# [프로젝트 추가 기록] 작은 청크와 안내도 부품 항목의 글자 색인, 질문 목적 규칙으로 검색 후보를 선택합니다.
# 추가 이유·한계·수업 근거 전체 목록: src/zzong_santafe_lag/learning_additions.md
from pathlib import Path
import json
import io
import contextlib
import time

project_folder = Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
reusable_names = [
    "full_parent_records", "full_search_chunks", "parent_lookup", "parent_ids",
    "chunk_vectors", "embedding_model", "token_count", "token_budget",
    "rank_candidates", "read_context", "evaluation_cases", "query_vectors",
]
if all(name in globals() for name in reusable_names):
    # 11번을 끝낸 같은 커널이면 벡터를 다시 계산하지 않습니다.
    preparation = {name: globals()[name] for name in reusable_names}
    print("같은 커널의 11번 메모리 자료를 재사용합니다.")
else:
    # 새 커널이면 앞 실험을 다시 실행합니다. 진행 상황은 임베딩 셀에서 표시합니다.
    source_path = project_folder / "notebooks" / "11_oil_table_and_full_embedding.ipynb"
    source_notebook = json.loads(source_path.read_text(encoding="utf-8"))
    preparation = {}
    for index, cell in enumerate(source_notebook["cells"]):
        if cell["cell_type"] != "code":
            continue
        if index == 9:
            exec(compile("".join(cell["source"]), f"11:cell_{index}", "exec"), preparation)
        else:
            with contextlib.redirect_stdout(io.StringIO()):
                exec(compile("".join(cell["source"]), f"11:cell_{index}", "exec"), preparation)
    print("새 커널에서 11번 준비와 전체 임베딩을 완료했습니다.")

for name in reusable_names:
    globals()[name] = preparation[name]

import re
import numpy as np
from collections import Counter, defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer

assert chunk_vectors.shape == (len(full_search_chunks), 768)
assert len(full_search_chunks) == 2213
assert all(token_count(c["content"]) <= token_budget for c in full_search_chunks)
parent_snapshot = json.dumps(full_parent_records, ensure_ascii=False, sort_keys=True)
print("원문:", len(parent_lookup), "| 청크:", len(full_search_chunks), "| 벡터:", chunk_vectors.shape)


임베딩 64/2213 | 오류 0 | 경과 16.9초 | 예상 남은 시간 566.6초
임베딩 128/2213 | 오류 0 | 경과 26.6초 | 예상 남은 시간 433.0초
임베딩 192/2213 | 오류 0 | 경과 31.7초 | 예상 남은 시간 333.2초
임베딩 256/2213 | 오류 0 | 경과 36.5초 | 예상 남은 시간 279.0초
임베딩 320/2213 | 오류 0 | 경과 41.3초 | 예상 남은 시간 244.3초
임베딩 384/2213 | 오류 0 | 경과 46.1초 | 예상 남은 시간 219.7초
임베딩 448/2213 | 오류 0 | 경과 51.1초 | 예상 남은 시간 201.3초
임베딩 512/2213 | 오류 0 | 경과 55.8초 | 예상 남은 시간 185.5초
임베딩 576/2213 | 오류 0 | 경과 60.8초 | 예상 남은 시간 172.8초
임베딩 640/2213 | 오류 0 | 경과 65.8초 | 예상 남은 시간 161.6초
임베딩 704/2213 | 오류 0 | 경과 71.0초 | 예상 남은 시간 152.3초
임베딩 768/2213 | 오류 0 | 경과 79.3초 | 예상 남은 시간 149.3초
임베딩 832/2213 | 오류 0 | 경과 85.3초 | 예상 남은 시간 141.5초
임베딩 896/2213 | 오류 0 | 경과 90.0초 | 예상 남은 시간 132.4초
임베딩 960/2213 | 오류 0 | 경과 94.7초 | 예상 남은 시간 123.7초
임베딩 1024/2213 | 오류 0 | 경과 100.0초 | 예상 남은 시간 116.1초
임베딩 1088/2213 | 오류 0 | 경과 105.3초 | 예상 남은 시간 108.8초
임베딩 1152/2213 | 오류 0 | 경과 110.5초 | 예상 남은 시간 101.7초
임베딩 1216/2213 | 오류 0 | 경과 115.7초 | 예상 남은 시간 94.9초
임베딩 1280/2213 | 오류 0 | 경과 121.0초 | 예상 남은 시간 88.2초
임베딩 1344/2213

## 1. 작은 조각과 안내도 항목의 검색어 색인

글자 2~4개씩 묶어 검색합니다. 표의 +, -와 숫자도 유지합니다.
안내도 참조 쪽수를 부품 이름처럼 검색하지 않도록 이름·안내도 제목만 색인합니다.


In [2]:
# [프로젝트 추가 기록] 작은 청크와 안내도 부품 항목의 글자 색인, 질문 목적 규칙으로 검색 후보를 선택합니다.
# 추가 이유·한계·수업 근거 전체 목록: src/zzong_santafe_lag/learning_additions.md
def normalize_query_text(text):
    """공백 차이를 줄이고 한글·영문·숫자·조작 기호를 남깁니다."""
    return "".join(re.findall(r"[가-힣A-Za-z0-9ℓ+\-]+", text)).lower()
assert normalize_query_text("ℓ") == "ℓ"

chunk_keyword_index = TfidfVectorizer(
    analyzer="char", ngram_range=(2, 4), dtype=np.float32
)
chunk_keyword_matrix = chunk_keyword_index.fit_transform(
    [normalize_query_text(c["content"]) for c in full_search_chunks]
)

navigation_rows = []
for parent_id, parent in parent_lookup.items():
    for item in parent["metadata"].get("navigation_items", []):
        navigation_rows.append({
            "parent_id": parent_id, "number": item["number"], "name": item["name"],
            "index_text": normalize_query_text(parent["metadata"]["title"] + " " + item["name"]),
        })
navigation_keyword_index = TfidfVectorizer(
    analyzer="char", ngram_range=(2, 4), dtype=np.float32
)
navigation_keyword_matrix = navigation_keyword_index.fit_transform(
    [row["index_text"] for row in navigation_rows]
)
indices_by_parent = defaultdict(list)
for index, chunk in enumerate(full_search_chunks):
    indices_by_parent[chunk["metadata"]["parent_record_id"]].append(index)

navigation_names = {
    normalize_query_text(item["name"])
    for parent in parent_lookup.values()
    for item in parent["metadata"].get("navigation_items", [])
}
# 드라이브 모드 값도 정답 질문에서 가져오지 않고 확인된 표의 열에서 읽습니다.
table_mode_values = {
    normalize_query_text(row["drive_mode"])
    for parent in parent_lookup.values()
    for row in parent["metadata"].get("table_rows", [])
    if "drive_mode" in row
}
print("청크 색인:", chunk_keyword_matrix.shape)
print("안내도 번호 항목:", len(navigation_rows))
print("표의 모드 값:", sorted(table_mode_values))


청크 색인: (2213, 218041)
안내도 번호 항목: 71
표의 모드 값: ['eco', 'sport']


## 2. 질문 목적 판단

명시적인 안내도·번호 질문은 안내도를 찾습니다.
일반적인 위치 질문은 자료에 있는 부품 이름도 함께 확인합니다.
용량·규격·점도·각주 질문과 표에 있는 모드의 기능 질문은 표를 찾습니다.
조작·주의·제한 질문은 설명문을 찾습니다.

여러 목적이 섞이면 해당 자료 유형을 합쳐 검색합니다.
명확한 유형을 판단하지 못하면 전체 자료에서 검색합니다.
이 규칙은 단어를 이용하는 첫 실험이며, 모든 한국어 표현을 이해하는 분류기는 아닙니다.


In [3]:
# [프로젝트 추가 기록] 작은 청크와 안내도 부품 항목의 글자 색인, 질문 목적 규칙으로 검색 후보를 선택합니다.
# 추가 이유·한계·수업 근거 전체 목록: src/zzong_santafe_lag/learning_additions.md
def has_any(text, phrases):
    """질문에 정해진 목적 표현이 있는지 확인합니다."""
    normalized = [normalize_query_text(phrase) for phrase in phrases]
    # 빈 표현은 모든 질문에 포함되므로 반드시 제외합니다.
    return any(phrase and phrase in text for phrase in normalized)

def classify_question(question):
    """질문 표현과 자료 구조를 보고 필요한 자료 유형과 판단 이유를 돌려줍니다."""
    text = normalize_query_text(question)
    explicit_navigation = has_any(text, ["안내도", "몇 번", "어느 항목", "번호로 표시", "번호가 표시"])
    location_words = has_any(text, ["위치", "어디", "자리"])
    operation_words = has_any(text, [
        "사용", "조작", "작동", "방법", "당기", "누르", "눌", "기능", "변속",
        "문제", "주의", "제한", "왜", "이유", "안 되는", "하면 안",
        "할 수 없", "변경", "상황", "언제", "어떻게", "밀어",
    ])
    known_component = any(name and name in text for name in navigation_names)
    quantity_words = has_any(text, [
        "용량", "규격", "추천 사양", "점도", "SAE", "각주", "리터", "ℓ",
        "표를", "표의", "표에", "온도 눈금",
    ])
    mode_function = (
        has_any(text, ["모드"]) and any(mode in text for mode in table_mode_values)
        and not explicit_navigation
    )
    intents, reasons = [], []
    if explicit_navigation or (location_words and known_component):
        intents.append("navigation")
        reasons.append("안내도·번호 표현 또는 자료에 있는 부품의 위치 질문")
    if quantity_words or mode_function:
        intents.append("table")
        reasons.append("용량·규격·각주 표현 또는 표의 모드 기능 질문")
    # 명시적으로 위치만 묻는 경우 일반적인 '버튼' 등의 표현은 조작으로 판단하지 않습니다.
    if operation_words and not (explicit_navigation and not has_any(text, ["방법", "사용", "조작"])):
        if not mode_function or has_any(text, ["제한", "주의", "왜", "안 되는"]):
            intents.append("explanation")
            reasons.append("사용·조작·주의·제한 질문")
    if not intents:
        intents, reasons = ["general"], ["유형이 명확하지 않아 전체 자료 검색"]
    return {
        "intents": intents, "reasons": reasons,
        "inside_vehicle_only": "navigation" in intents and has_any(text, ["차량 내부", "내부 안내도"]),
    }

def parent_matches_intent(parent, decision):
    """기록된 자료 유형으로 검색 범위를 고릅니다. 원문 ID나 평가 정답은 사용하지 않습니다."""
    meta = parent["metadata"]
    kind = meta["content_type"]
    for intent in decision["intents"]:
        if intent == "general":
            return True
        if intent == "navigation" and kind == "vehicle_overview_navigation":
            if decision["inside_vehicle_only"] and "차량 내부" not in meta["title"]:
                continue
            return True
        if intent == "table" and (
            kind in {"structured_table", "table_with_images"} or bool(meta.get("footnotes"))
        ):
            return True
        if intent == "explanation" and kind in {"instruction_text", "figure_with_text"}:
            return True
    return False

# 기대 원문 ID를 쓰지 않고 질문 표현만으로 판단한 예시입니다.
for question in [
    "패들 쉬프트는 차량 내부 안내도에서 몇 번인가요?",
    "엔진 오일 용량과 규격은?",
    "패들 쉬프트를 동시에 당기면 변속되나요?",
    "시동 버튼 위치와 사용하는 방법을 알려줘.",
    "차대번호는 어디에 새겨져 있나요?",
    "오늘 서울 날씨는?",
]:
    print(question, "→", classify_question(question)["intents"])

# 단위만 남은 표현과 빈 표현 때문에 모든 질문이 표로 분류되는 것을 막습니다.
assert not has_any(normalize_query_text("날씨"), [""])
assert classify_question("용량 4.8 ℓ")["intents"] == ["table"]


패들 쉬프트는 차량 내부 안내도에서 몇 번인가요? → ['navigation']
엔진 오일 용량과 규격은? → ['table']
패들 쉬프트를 동시에 당기면 변속되나요? → ['explanation']
시동 버튼 위치와 사용하는 방법을 알려줘. → ['navigation', 'explanation']
차대번호는 어디에 새겨져 있나요? → ['general']
오늘 서울 날씨는? → ['general']


## 3. 청크 검색과 자료 유형 선택

청크의 의미 점수와 검색어 일치 점수를 각각 50% 사용합니다.
안내도는 짧은 번호 항목의 이름 점수도 사용합니다.
유형이 정해지면 해당 범위에서 순위를 매기고, 범위가 비면 전체 검색으로 되돌아갑니다.
범위가 상위 결과 개수보다 작으면 전체 후보를 뒤에 덧붙이며 이 사실을 표시합니다.

검토 상태는 순위를 올리는 보너스로 사용하지 않습니다. 결과에 상태를 그대로 붙입니다.


In [4]:
# [프로젝트 추가 기록] 작은 청크와 안내도 부품 항목의 글자 색인, 질문 목적 규칙으로 검색 후보를 선택합니다.
# 추가 이유·한계·수업 근거 전체 목록: src/zzong_santafe_lag/learning_additions.md
def purpose_rank(question, question_vector, top_k=3, use_routing=True):
    """작은 조각의 결합 점수로 부모를 정렬하고 질문 목적에 맞는 범위를 선택합니다."""
    if not 1 <= top_k <= len(parent_lookup):
        raise ValueError("검색 결과 수를 확인하세요.")
    semantic_scores = chunk_vectors @ question_vector
    query_terms = chunk_keyword_index.transform([normalize_query_text(question)])
    keyword_scores = (chunk_keyword_matrix @ query_terms.T).toarray().ravel()
    combined_scores = 0.5 * semantic_scores + 0.5 * keyword_scores

    # 안내도 긴 목록 전체 대신 각각의 부품 이름을 비교합니다.
    row_terms = navigation_keyword_index.transform([normalize_query_text(question)])
    row_scores = (navigation_keyword_matrix @ row_terms.T).toarray().ravel()
    best_navigation_rows = {}
    for row, score in zip(navigation_rows, row_scores):
        parent_id = row["parent_id"]
        if parent_id not in best_navigation_rows or score > best_navigation_rows[parent_id]["score"]:
            best_navigation_rows[parent_id] = {"score": float(score), "row": row}

    all_hits = []
    for parent_id, indices in indices_by_parent.items():
        best_index = max(indices, key=lambda index: float(combined_scores[index]))
        score = float(combined_scores[best_index])
        matched_row = None
        if parent_id in best_navigation_rows:
            # 번호 항목의 이름 점수와 해당 부모의 최고 의미 점수를 같은 비중으로 씁니다.
            best_semantic_index = max(indices, key=lambda index: float(semantic_scores[index]))
            navigation_score = 0.5 * float(semantic_scores[best_semantic_index]) + (
                0.5 * best_navigation_rows[parent_id]["score"]
            )
            if navigation_score > score:
                score = navigation_score
                best_index = best_semantic_index
                matched_row = best_navigation_rows[parent_id]["row"]
        all_hits.append({
            "score": score, "matched_chunk": full_search_chunks[best_index],
            "parent_record": parent_lookup[parent_id], "matched_navigation_row": matched_row,
        })
    all_hits.sort(key=lambda hit: hit["score"], reverse=True)

    decision = classify_question(question) if use_routing else {
        "intents": ["general"], "reasons": ["유형 선택 없이 전체 검색"],
        "inside_vehicle_only": False,
    }
    preferred = [hit for hit in all_hits if parent_matches_intent(hit["parent_record"], decision)]
    used_fallback = not preferred
    if not preferred:
        preferred = all_hits
    selected = preferred[:top_k]
    if len(selected) < top_k:
        seen = {hit["parent_record"]["metadata"]["record_id"] for hit in selected}
        selected.extend(hit for hit in all_hits if hit["parent_record"]["metadata"]["record_id"] not in seen)
        selected = selected[:top_k]
        used_fallback = True
    for hit in selected:
        hit["routing"] = decision
        hit["preferred_candidate_count"] = len(preferred)
        hit["used_fallback"] = used_fallback
        hit["verification_status"] = hit["parent_record"]["metadata"]["verification_status"]
    return selected

def purpose_search(question, top_k=3):
    """질문을 벡터로 바꾸고 목적에 따른 검색 후보와 선택 이유를 가져옵니다."""
    if not question.strip() or token_count(question) > token_budget:
        raise ValueError("짧고 비어 있지 않은 질문을 입력하세요.")
    vector = embedding_model.encode(
        question, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False
    )
    return purpose_rank(question, vector, top_k=top_k)


## 4. 같은 30개 질문 비교

질문과 기대 출처를 바꾸지 않고 세 방식을 비교합니다.
유형 선택을 적용하기 전의 결과도 남겨, 작은 조각 색인과 유형 선택의 효과를 구분합니다.
개선뿐 아니라 악화된 질문과 상위3에 기대 출처가 없는 질문도 출력합니다.


In [5]:
# [프로젝트 추가 기록] 작은 청크와 안내도 부품 항목의 글자 색인, 질문 목적 규칙으로 검색 후보를 선택합니다.
# 추가 이유·한계·수업 근거 전체 목록: src/zzong_santafe_lag/learning_additions.md
comparison_results = []
for case, vector in zip(evaluation_cases, query_vectors):
    expected = set(case["expected_ids"])
    row = dict(case)
    methods = {
        "baseline": rank_candidates(case["question"], vector, method="hybrid", top_k=3),
        "chunk": purpose_rank(case["question"], vector, use_routing=False, top_k=3),
        "routed": purpose_rank(case["question"], vector, use_routing=True, top_k=3),
    }
    for method, hits in methods.items():
        ids = [hit["parent_record"]["metadata"]["record_id"] for hit in hits]
        row[method + "_ids"] = ids
        row[method + "_top1"] = ids[0] in expected
        row[method + "_top3"] = bool(expected.intersection(ids))
        assert len(ids) == len(set(ids))
    row["routing"] = methods["routed"][0]["routing"]
    row["candidate_count"] = methods["routed"][0]["preferred_candidate_count"]
    row["fallback"] = methods["routed"][0]["used_fallback"]
    comparison_results.append(row)

# 11번과 같은 질문·기대 출처·비교 조건을 유지합니다.
assert len(comparison_results) == 30
assert sum(r["baseline_top1"] for r in comparison_results) == 14
assert sum(r["baseline_top3"] for r in comparison_results) == 21

for group in ["기존 확인 질문", "이번 원본 확인 질문", "전체"]:
    rows = [r for r in comparison_results if r["group"] == group] if group != "전체" else comparison_results
    print("\n", group, "| 질문 수:", len(rows))
    for method in ["baseline", "chunk", "routed"]:
        print(method, "| 첫 결과:", sum(r[method + "_top1"] for r in rows),
              "| 상위3:", sum(r[method + "_top3"] for r in rows))
print("\n유형 판단:", dict(Counter(",".join(r["routing"]["intents"]) for r in comparison_results)))
print("유형별 후보 수:", sorted(set(r["candidate_count"] for r in comparison_results)))
print("\n첫 결과 개선·악화:")
for row in comparison_results:
    if row["baseline_top1"] != row["routed_top1"]:
        status = "개선" if row["routed_top1"] else "악화"
        print(status, "|", row["question"], "| 유형:", row["routing"]["intents"])
        print(" 기존:", row["baseline_ids"][0], "→ 보완:", row["routed_ids"][0])
print("\n보완 후 상위3에 지정 출처가 없는 질문:")
for row in comparison_results:
    if not row["routed_top3"]:
        print(row["question"], "| 기대:", row["expected_ids"], "| 후보:", row["routed_ids"])



 기존 확인 질문 | 질문 수: 20
baseline | 첫 결과: 6 | 상위3: 12
chunk | 첫 결과: 16 | 상위3: 18
routed | 첫 결과: 17 | 상위3: 18

 이번 원본 확인 질문 | 질문 수: 10
baseline | 첫 결과: 8 | 상위3: 9
chunk | 첫 결과: 5 | 상위3: 9
routed | 첫 결과: 9 | 상위3: 10

 전체 | 질문 수: 30
baseline | 첫 결과: 14 | 상위3: 21
chunk | 첫 결과: 21 | 상위3: 27
routed | 첫 결과: 26 | 상위3: 28

유형 판단: {'navigation': 12, 'table': 9, 'general': 1, 'explanation': 7, 'table,explanation': 1}
유형별 후보 수: [3, 5, 518, 522, 527]

첫 결과 개선·악화:
개선 | 실외 미러를 접는 버튼은 차량 내부 안내도의 어디에 있나요? | 유형: ['navigation']
 기존: auto_topic_150 → 보완: overview_33
개선 | 후드 열림 레버의 위치를 보여주는 안내도를 찾아줘. | 유형: ['navigation']
 기존: auto_topic_163 → 보완: overview_33
개선 | 지문 인증 시스템은 차량 내부 안내도에서 몇 번인가요? | 유형: ['navigation']
 기존: auto_topic_114 → 보완: overview_33
개선 | 유리창 잠금 버튼과 전자식 차일드 락 버튼 위치를 알려줘. | 유형: ['navigation']
 기존: auto_topic_126 → 보완: overview_33
개선 | 시동 버튼 위치를 보여주는 안내도가 필요해. | 유형: ['navigation']
 기존: auto_topic_271 → 보완: overview_34
개선 | 스마트폰 무선 충전 표시등은 내부 안내도의 어느 항목인가요? | 유형: ['navigation']
 기존: auto_topic_

## 5. 연결된 원문·각주·그림 참조 함께 읽기

점수가 가장 높은 조각만 답변에 사용하지 않습니다.
부모 원문과 필수 연결 기록을 함께 읽고, 그림 참조와 검토 표시를 반환합니다.
아직 확인하지 않은 그림·원문은 확인된 자료처럼 표시하지 않습니다.


In [6]:
# [프로젝트 추가 기록] 작은 청크와 안내도 부품 항목의 글자 색인, 질문 목적 규칙으로 검색 후보를 선택합니다.
# 추가 이유·한계·수업 근거 전체 목록: src/zzong_santafe_lag/learning_additions.md
def build_evidence_package(hit):
    """검색 후보의 부모·각주·주의사항·그림 참조와 검토 상태를 함께 묶습니다."""
    parent_id = hit["parent_record"]["metadata"]["record_id"]
    context = read_context(parent_id)
    images, seen_images = [], set()
    for record in context:
        for part in record["image_parts"]:
            identity = (
                part["pdf_page_number"],
                json.dumps(part.get("pdf_image_key", part.get("name")), ensure_ascii=False),
            )
            if identity not in seen_images:
                seen_images.add(identity)
                images.append(part)
    pending_ids = [
        record["metadata"]["record_id"] for record in context
        if record["metadata"]["verification_status"] == "auto_draft_needs_review"
    ]
    continuation_ids = [
        record_id for record in context
        for record_id in record["metadata"].get("unreviewed_continuation_record_ids", [])
    ]
    return {
        "matched_chunk": hit["matched_chunk"], "context_records": context, "image_parts": images,
        "routing": hit["routing"], "pending_review_record_ids": pending_ids,
        "unreviewed_continuation_record_ids": list(dict.fromkeys(continuation_ids)),
        "answer_generation_status": "not_implemented",
    }

demo_questions = [
    "패들 쉬프트 위치를 보여주는 내부 안내도를 찾아줘.",
    "엔진 오일의 용량과 추천 사양을 알려줘.",
    "패들 쉬프트를 동시에 당기면 변속되나요?",
]
demo_packages = []
for question in demo_questions:
    hit = purpose_search(question, top_k=1)[0]
    package = build_evidence_package(hit)
    demo_packages.append(package)
    print("\n질문:", question)
    print("유형:", hit["routing"]["intents"], "| 이유:", hit["routing"]["reasons"])
    print("첫 원문:", hit["parent_record"]["metadata"]["title"],
          "| PDF", hit["parent_record"]["metadata"]["source_pages"])
    print("함께 읽는 원문:", [(r["metadata"]["title"], r["metadata"]["source_pages"])
        for r in package["context_records"]])
    print("그림 참조:", [(p["pdf_page_number"], p.get("name"), p.get("linkage_status", "개별 연결 상태 미기록"))
        for p in package["image_parts"]])
    print("검토가 필요한 원문:", package["pending_review_record_ids"])
    print("추가 검토가 필요한 이어지는 기록:", package["unreviewed_continuation_record_ids"])

assert json.dumps(full_parent_records, ensure_ascii=False, sort_keys=True) == parent_snapshot
print("\n원문·청크 내용 변경:", "없음")
print("생성 답변·파일·Supabase 저장:", "실행하지 않음")



질문: 패들 쉬프트 위치를 보여주는 내부 안내도를 찾아줘.
유형: ['navigation'] | 이유: ['안내도·번호 표현 또는 자료에 있는 부품의 위치 질문']
첫 원문: 차량 내부 III | PDF [35, 36]
함께 읽는 원문: [('차량 내부 III', [35, 36])]
그림 참조: [(35, 'I1.jpg', '개별 연결 상태 미기록')]
검토가 필요한 원문: []
추가 검토가 필요한 이어지는 기록: []

질문: 엔진 오일의 용량과 추천 사양을 알려줘.
유형: ['table'] | 이유: ['용량·규격·각주 표현 또는 표의 모드 기능 질문']
첫 원문: 추천 오일 및 용량 — 각주 및 주의사항 | PDF [43]
함께 읽는 원문: [('추천 오일 및 용량 — 각주 및 주의사항', [43]), ('추천 오일 및 용량', [43])]
그림 참조: []
검토가 필요한 원문: []
추가 검토가 필요한 이어지는 기록: []

질문: 패들 쉬프트를 동시에 당기면 변속되나요?
유형: ['explanation'] | 이유: ['사용·조작·주의·제한 질문']
첫 원문: 패들 쉬프트 (수동 변속 모드) | PDF [397]
함께 읽는 원문: [('패들 쉬프트 (수동 변속 모드)', [397]), ('드라이브 모드 별 패들 쉬프트 레버 기능', [398]), ('회생 제동 시스템 사용 제한', [402]), ('주행 모드별 패들 쉬프트의 기능', [402])]
그림 참조: [(397, 'I2.jpg', 'visually_verified')]
검토가 필요한 원문: []
추가 검토가 필요한 이어지는 기록: ['unreviewed_paddle_limit_403']

원문·청크 내용 변경: 없음
생성 답변·파일·Supabase 저장: 실행하지 않음


## 6. 범위를 잘못 좁힐 수 있는 질문 확인

아래 질문에는 정답 출처를 지정하지 않습니다. 분류와 후보를 관찰하는 예시입니다.
혼합 질문, 표현이 다른 질문, 자료 밖 질문에서는 이 간단한 규칙의 한계가 드러날 수 있습니다.
자료 유형 선택이 올바른 답변을 보장하지는 않습니다.


In [7]:
# [프로젝트 추가 기록] 작은 청크와 안내도 부품 항목의 글자 색인, 질문 목적 규칙으로 검색 후보를 선택합니다.
# 추가 이유·한계·수업 근거 전체 목록: src/zzong_santafe_lag/learning_additions.md
boundary_questions = [
    "시동 버튼은 어디에 있고 어떻게 사용하나요?",
    "차대번호는 어디에 새겨져 있나요?",
    "냉각수 보조 탱크 캡을 언제 열 수 있나요?",
    "0W-20의 표에 표시된 온도 눈금은?",
    "오늘 서울 날씨와 내일 강수확률은?",
    "싼타페 중고차의 현재 시세는 얼마인가요?",
]
for question in boundary_questions:
    hit = purpose_search(question, top_k=1)[0]
    parent = hit["parent_record"]
    print(question, "→", hit["routing"]["intents"],
          "| 후보:", parent["metadata"]["title"], "| 상태:", hit["verification_status"])
print("\n날씨·시세 질문에도 후보가 나올 수 있으므로 검색 점수만으로 답변하면 안 됩니다.")
print("원문 검토 필요 상태:", sum(p["metadata"]["verification_status"] == "auto_draft_needs_review"
    for p in full_parent_records), "개")


시동 버튼은 어디에 있고 어떻게 사용하나요? → ['navigation', 'explanation'] | 후보: 차량 내부 II | 상태: sample_verified
차대번호는 어디에 새겨져 있나요? → ['general'] | 후보: 차대번호 (VIN) | 상태: sample_verified
냉각수 보조 탱크 캡을 언제 열 수 있나요? → ['explanation'] | 후보: 냉각수량 점검 및 보충 | 상태: auto_draft_needs_review
0W-20의 표에 표시된 온도 눈금은? → ['table'] | 후보: 온도에 따른 엔진 오일 SAE 점도 분류표 | 상태: sample_verified
오늘 서울 날씨와 내일 강수확률은? → ['general'] | 후보: 폭설 시 행동요령 | 상태: auto_draft_needs_review
싼타페 중고차의 현재 시세는 얼마인가요? → ['general'] | 후보: 중고차 구입/디지털 키 관련 차량 정비 | 상태: auto_draft_needs_review

날씨·시세 질문에도 후보가 나올 수 있으므로 검색 점수만으로 답변하면 안 됩니다.
원문 검토 필요 상태: 507 개


## 다음 순서

이번 규칙의 비교 결과와 잘못 좁힌 질문을 확인합니다.
검색 후보의 확인 상태와 연결된 주의사항을 유지하면서 개인 .py 모듈로 옮길 준비를 합니다.
답변 가능 여부 판단과 챗봇 화면, 개인 Supabase 저장은 이후 단계입니다.


## 최종 실행 기록 — 2026-10-01

분류 예시를 확인하면서 빈 단위 표현이 모든 질문에 일치하는 오류를 수정했습니다.
차대번호의 '번호'를 안내도 번호로 오해하지 않도록 명시적인 안내도 표현만 사용했습니다.
위치·사용 방법을 함께 묻는 질문은 두 유형을 함께 검색합니다.
아래 결과는 수정한 코드 전체를 다시 실행한 결과입니다.

| 질문 묶음 | 방식 | 첫 결과에 지정 출처 | 상위3에 지정 출처 |
|---|---|---:|---:|
| 기존 20개 | 기존 결합 | 6 | 12 |
| 기존 20개 | 청크·번호 항목 결합 | 16 | 18 |
| 기존 20개 | 유형 선택 적용 | 17 | 18 |
| 새 10개 | 기존 결합 | 8 | 9 |
| 새 10개 | 청크·번호 항목 결합 | 5 | 9 |
| 새 10개 | 유형 선택 적용 | 9 | 10 |
| 전체 30개 | 기존 결합 | 14 | 21 |
| 전체 30개 | 청크·번호 항목 결합 | 21 | 27 |
| 전체 30개 | 유형 선택 적용 | 26 | 28 |

질문과 기대 출처, 결합 비중 50%·50%는 그대로 유지했습니다.
이 수치는 지정한 원문 ID를 찾은 횟수이며 생성 답변의 정확도가 아닙니다.
유형별 후보 수는 3·5·518·522·527개였습니다. 특히 내부 안내도 후보가 3개일 때
상위3 포함만으로 충분히 잘 찾았다고 판단하면 안 되므로 첫 결과도 함께 기록했습니다.

### 남은 확인 사항

- '안전벨트 버클을 얼마나 밀어 넣어야 하나요?'와 '차단 클립을 버클에 꽂으면 경고등과 경고음은 어떻게 되나요?'에서 지정한 54쪽 기록이 상위3에 없었습니다.
- 반환된 다른 안전벨트 상세 기록에 같은 답의 근거가 있는지 확인하기 전에는 대체 출처를 정답으로 인정하지 않습니다.
- 유형 선택은 현재 분류된 표와 안내도를 대상으로 합니다. 미분류 자동 초안의 표가 검색 범위에서 빠질 수 있습니다.
- 혼합 질문은 두 유형을 합쳐 검색하지만 각각의 질문을 모두 만족하는 근거를 모았는지 아직 자동 판단하지 않습니다.
- 자료 밖 날씨·시세 질문에도 후보가 반환됩니다. 자동 답변·거절 판정은 아직 구현하지 않았습니다.
- 원문 검토 필요 상태 507개와 이미지 참조의 개별 연결 상태를 유지했습니다.

### 완료 범위와 저장

2,213개 벡터를 다시 계산했고 실패는 0개입니다. 이번 임베딩 계산 시간은 293.6초입니다.
원문·청크 내용은 바꾸지 않았으며, 원문·각주·주의사항·그림 참조를 묶어 읽는 예제를 확인했습니다.
노트북 코드와 실행 결과만 저장했습니다. 데이터·벡터 파일 및 Supabase 저장은 하지 않았습니다.
같은 커널의 11번 자료가 있으면 재사용하고, 새 커널에서는 임베딩을 다시 계산합니다.

다음은 남은 안전벨트 질문의 근거를 확인하고, 개인 .py 모듈로 준비·청킹·임베딩·검색 함수를 분리하는 단계입니다.
